# Benchmark Runner

Runs three link-prediction methods and saves `full_results.json` per experiment.
**Learnable PPR** is run separately via `learnable_ppr.ipynb` — not duplicated here.

| Method | Subgraph | Sweep | Output |
|--------|----------|-------|--------|
| Full Graph | None | — | `results/benchmark/{ds}/{enc}/` |
| Static PPR | PPR + sweep cut | `PPR_ALPHAS x PPR_EPSILONS` | `results/benchmark-ppr/{ds}/{enc}_a{a}_e{e}/` |
| Static k-hop | k-hop | `K_VALUES` | `results/benchmark-khop/{ds}/{enc}_k{k}/` |

After running this + `learnable_ppr.ipynb`, open `benchmark_analysis.ipynb` to compare.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════════

SEED = 42
DATASETS  = ['FB15K237']           # FB15K237, WN18RR, NELL-995
ENCODERS  = ['GCN', 'SAGE', 'GAT']

# Shared architecture
FEATURE_DIM     = 128
HIDDEN_CHANNELS = 256
NUM_LAYERS      = 3
DROPOUT         = 0.3

# Full-graph training
FULL_EPOCHS     = 500
FULL_BATCH_SIZE = 4096
FULL_LR         = 0.005
FULL_PATIENCE   = 30

# Subgraph-based training (PPR, k-hop)
SUB_EPOCHS          = 500
SUB_BATCH_SIZE      = 512       # subgraphs per GNN forward
SUB_LR              = 0.005
SUB_PATIENCE        = 30
EDGES_PER_EPOCH     = 100000    # subsample training edges (None = all)
EVAL_STEPS          = 5
WEIGHT_DECAY        = 1e-5
GRAD_CLIP           = 1.0

# Static PPR sweep (LCILP-style: approximate PPR + conductance sweep cut)
PPR_ALPHAS    = [0.80, 0.85, 0.90]   # teleportation / damping values
PPR_EPSILONS  = [1e-2, 1e-3, 1e-4]   # PPR precision thresholds
PPR_WINDOW    = 10                    # sweep cut early-stop window

# Static k-hop sweep
K_VALUES = [1, 2, 3]

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}  |  Datasets: {DATASETS}  |  Encoders: {ENCODERS}')

In [ ]:
import numpy as np, random, json, os, time

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

from subgrapher.benchmark.data_prep import prepare_link_prediction_data
from subgrapher.utils.models import GCN, SAGE, GAT, LinkPredictor


DATASET_PATHS = {
    'FB15K237':  'data/FB15K237/train.txt',
    'WN18RR':    'data/WN18RR/train.txt',
    'NELL-995':  'data/NELL-995/train.txt',
}

def make_encoder(enc_type, in_ch=FEATURE_DIM, hid=HIDDEN_CHANNELS,
                 n_layers=NUM_LAYERS, dropout=DROPOUT):
    if enc_type == 'GCN':  return GCN(in_ch, hid, hid, n_layers, dropout)
    if enc_type == 'SAGE': return SAGE(in_ch, hid, hid, n_layers, dropout)
    if enc_type == 'GAT':  return GAT(in_ch, hid, hid, n_layers, dropout, heads=4)
    raise ValueError(enc_type)

def make_predictor(hid=HIDDEN_CHANNELS, n_layers=NUM_LAYERS, dropout=DROPOUT):
    return LinkPredictor(hid, hid, 1, n_layers, dropout)

def ensure_train_only_edges(data, dd):
    """Point data.edge_index at the train-only edge set (idempotent).

    `prepare_link_prediction_data` returns a PyG Data whose edge_index contains
    all positives (train + val + test). Extracting a subgraph around a val/test
    edge (u, v) with the full graph would trivially include (u, v) itself and
    leak the target into message passing (classic signature: SAGE val_mrr ~ 1.0).
    Every downstream method (Full Graph, Static PPR, k-hop, Learnable PPR)
    should call this first. Safe to call multiple times.
    """
    if not getattr(data, '_edge_index_train_only', False):
        data._orig_edge_index = data.edge_index
        data.edge_index = dd['train_edge_index']
        data._edge_index_train_only = True
        print(f'  [train-only edges] swapped: '
              f"{data._orig_edge_index.size(1):,} -> {data.edge_index.size(1):,}")
    return data

def save_full_results(base_dir, result_dict):
    """Save full_results.json inside a timestamped run directory.
    Layout: base_dir/runs/{timestamp}/full_results.json
    Also writes base_dir/full_results.json as the latest copy.
    """
    run_id = time.strftime('%Y%m%d_%H%M%S')
    result_dict['run_id'] = run_id
    result_dict['timestamp'] = time.strftime('%Y-%m-%d %H:%M:%S')

    run_dir = os.path.join(base_dir, 'runs', run_id)
    os.makedirs(run_dir, exist_ok=True)
    run_path = os.path.join(run_dir, 'full_results.json')
    with open(run_path, 'w') as f:
        json.dump(result_dict, f, indent=2, default=str)
    print(f'  -> {run_path}')

    os.makedirs(base_dir, exist_ok=True)
    latest_path = os.path.join(base_dir, 'full_results.json')
    with open(latest_path, 'w') as f:
        json.dump(result_dict, f, indent=2, default=str)
    print(f'  -> {latest_path} (latest)')

print('Ready.')

## 1. Load Datasets

In [ ]:
datasets = {}
for ds_name in DATASETS:
    print(f'\nLoading {ds_name}...')
    dd = prepare_link_prediction_data(
        DATASET_PATHS[ds_name],
        feature_method='random', feature_dim=FEATURE_DIM)
    datasets[ds_name] = dd
    data = dd['data']; se = dd['split_edge']
    print(f'  Nodes: {data.num_nodes:,}  Edges: {data.edge_index.size(1):,}')
    print(f'  Train: {se["train"]["source_node"].size(0):,}  '
          f'Val: {se["valid"]["source_node"].size(0):,}  '
          f'Test: {se["test"]["source_node"].size(0):,}')

## 1.5. Use Train-Only Edges for Message Passing (optional pre-swap)

`data.edge_index` from `prepare_link_prediction_data` contains **all** positive edges (train + val + test). Extracting a subgraph around a val/test edge `(u, v)` using this full graph would include `(u, v)` itself and leak the target into message passing (classic symptom: SAGE val_mrr ≈ 1.0 within 5 epochs).

This cell pre-swaps every dataset to train-only edges. **It is optional** — each method section below also calls `ensure_train_only_edges(...)` at its own start, so you can skip any section (e.g. run 1 → 1.5 → 3, or even 1 → 3) and still be safe. The swap is idempotent.

In [ ]:
for ds_name in DATASETS:
    print(f'[{ds_name}]')
    ensure_train_only_edges(datasets[ds_name]['data'], datasets[ds_name])

## 2. Full Graph (No Subgraph)

In [ ]:
from subgrapher.benchmark.trainer import benchmark_model

In [ ]:
for ds_name in DATASETS:
    dd = datasets[ds_name]
    data = dd['data']
    ensure_train_only_edges(data, dd)   # no-op if Section 1.5 already ran
    split_edge = dd['split_edge']

    for enc_type in ENCODERS:
        print(f'\n=== Full Graph: {ds_name} / {enc_type} ===')
        torch.manual_seed(SEED)
        encoder   = make_encoder(enc_type)
        predictor = make_predictor()
        encoder.reset_parameters()
        predictor.reset_parameters()

        result = benchmark_model(
            enc_type, encoder, predictor, data, split_edge,
            epochs=FULL_EPOCHS, batch_size=FULL_BATCH_SIZE,
            lr=FULL_LR, eval_steps=EVAL_STEPS, device=DEVICE,
            patience=FULL_PATIENCE, weight_decay=WEIGHT_DECAY,
            grad_clip=GRAD_CLIP)

        save_full_results(
            f'results/benchmark/{ds_name}/{enc_type}',
            {
                'dataset': ds_name,
                'encoder': enc_type,
                'test_results': {k: float(v) for k, v in result['test_results'].items()},
                'train_time': result['train_time'],
                'num_params': result['num_params'],
                'best_epoch': result['best_epoch'],
                'stopped_early': result['stopped_early'],
                'config': {
                    'epochs': FULL_EPOCHS,
                    'batch_size': FULL_BATCH_SIZE,
                    'lr': FULL_LR,
                    'patience': FULL_PATIENCE,
                    'hidden_channels': HIDDEN_CHANNELS,
                    'num_layers': NUM_LAYERS,
                    'dropout': DROPOUT,
                },
                'seed': SEED,
            })

        del encoder, predictor, result
        torch.cuda.empty_cache()

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()
print('Full Graph done — GPU memory released.')

## 3. Static PPR Subgraph

In [ ]:
from subgrapher.benchmark_ppr.ppr_extractor import StaticPPRExtractor
from subgrapher.benchmark_ppr.trainer_batched import train_model_ppr_batched
from subgrapher.benchmark_ppr.evaluator import evaluate_ppr

In [ ]:
for ds_name in DATASETS:
    dd = datasets[ds_name]
    data = dd['data']; split_edge = dd['split_edge']
    ensure_train_only_edges(data, dd)   # no-op if already swapped

    for ppr_alpha in PPR_ALPHAS:
        for ppr_eps in PPR_EPSILONS:
            ppr_ext = StaticPPRExtractor(
                data, alpha=ppr_alpha, epsilon=ppr_eps,
                window=PPR_WINDOW)

            for enc_type in ENCODERS:
                print(f'\n=== Static PPR (a={ppr_alpha}, e={ppr_eps}): '
                      f'{ds_name} / {enc_type} ===')
                torch.manual_seed(SEED)
                encoder   = make_encoder(enc_type)
                predictor = make_predictor()
                encoder.reset_parameters()
                predictor.reset_parameters()

                cache_dir = f'cache/benchmark-ppr/{ds_name}/a{ppr_alpha}_e{ppr_eps}'
                t0 = time.time()
                hist = train_model_ppr_batched(
                    encoder, predictor, data, split_edge, ppr_ext,
                    epochs=SUB_EPOCHS, batch_size=SUB_BATCH_SIZE,
                    lr=SUB_LR, eval_steps=EVAL_STEPS, device=DEVICE,
                    verbose=True, patience=SUB_PATIENCE,
                    weight_decay=WEIGHT_DECAY, grad_clip=GRAD_CLIP,
                    edges_per_epoch=EDGES_PER_EPOCH,
                    cache_dir=cache_dir)
                train_time = time.time() - t0

                test_res = evaluate_ppr(
                    encoder, predictor, data, split_edge, ppr_ext,
                    split='test', batch_size=SUB_BATCH_SIZE, device=DEVICE,
                    max_edges=5000, cache_dir=cache_dir)

                save_full_results(
                    f'results/benchmark-ppr/{ds_name}/{enc_type}_a{ppr_alpha}_e{ppr_eps}',
                    {
                        'dataset': ds_name,
                        'encoder': enc_type,
                        'ppr_alpha': ppr_alpha,
                        'ppr_epsilon': ppr_eps,
                        'ppr_window': PPR_WINDOW,
                        'test_results': {k: float(v) for k, v in test_res.items()},
                        'train_time': train_time,
                        'best_epoch': hist.get('best_epoch', 0),
                        'stopped_early': hist.get('stopped_early', False),
                        'config': {
                            'ppr_alpha': ppr_alpha,
                            'ppr_epsilon': ppr_eps,
                            'ppr_window': PPR_WINDOW,
                            'epochs': SUB_EPOCHS,
                            'batch_size': SUB_BATCH_SIZE,
                            'lr': SUB_LR,
                            'patience': SUB_PATIENCE,
                            'hidden_channels': HIDDEN_CHANNELS,
                            'num_layers': NUM_LAYERS,
                            'dropout': DROPOUT,
                        },
                        'seed': SEED,
                    })

                del encoder, predictor, hist, test_res
                torch.cuda.empty_cache()

            del ppr_ext

In [ ]:
gc.collect()
torch.cuda.empty_cache()
print('Static PPR done — GPU memory released.')

## 4. Static k-hop Subgraph

In [ ]:
from subgrapher.benchmark_khop.run_khop_benchmark import load_or_create_khop_preprocessor
from subgrapher.benchmark_khop.khop_extractor import StaticKHopExtractor
from subgrapher.benchmark_khop.trainer_batched import train_model_khop_batched
from subgrapher.benchmark_khop.evaluator import evaluate_khop

In [ ]:
for ds_name in DATASETS:
    dd = datasets[ds_name]
    data = dd['data']; split_edge = dd['split_edge']
    ensure_train_only_edges(data, dd)   # no-op if already swapped

    for k in K_VALUES:
        khop_pre = load_or_create_khop_preprocessor(ds_name, data, k)
        khop_ext = StaticKHopExtractor(data, k=k, preprocessor=khop_pre)

        for enc_type in ENCODERS:
            print(f'\n=== Static k-hop (k={k}): {ds_name} / {enc_type} ===')
            torch.manual_seed(SEED)
            encoder   = make_encoder(enc_type)
            predictor = make_predictor()
            encoder.reset_parameters()
            predictor.reset_parameters()

            cache_dir = f'cache/benchmark-khop/{ds_name}/k{k}'
            t0 = time.time()
            hist = train_model_khop_batched(
                encoder, predictor, data, split_edge, khop_ext,
                epochs=SUB_EPOCHS, batch_size=SUB_BATCH_SIZE,
                lr=SUB_LR, eval_steps=EVAL_STEPS, device=DEVICE,
                verbose=True, patience=SUB_PATIENCE,
                weight_decay=WEIGHT_DECAY, grad_clip=GRAD_CLIP,
                edges_per_epoch=EDGES_PER_EPOCH,
                cache_dir=cache_dir)
            train_time = time.time() - t0

            test_res = evaluate_khop(
                encoder, predictor, data, split_edge, khop_ext,
                split='test', batch_size=SUB_BATCH_SIZE, device=DEVICE,
                max_edges=5000, cache_dir=cache_dir)

            save_full_results(
                f'results/benchmark-khop/{ds_name}/{enc_type}_k{k}',
                {
                    'dataset': ds_name,
                    'encoder': enc_type,
                    'k': k,
                    'test_results': {k_: float(v) for k_, v in test_res.items()},
                    'train_time': train_time,
                    'best_epoch': hist.get('best_epoch', 0),
                    'stopped_early': hist.get('stopped_early', False),
                    'config': {
                        'k': k,
                        'epochs': SUB_EPOCHS,
                        'batch_size': SUB_BATCH_SIZE,
                        'lr': SUB_LR,
                        'patience': SUB_PATIENCE,
                        'hidden_channels': HIDDEN_CHANNELS,
                        'num_layers': NUM_LAYERS,
                        'dropout': DROPOUT,
                    },
                    'seed': SEED,
                })

            del encoder, predictor, hist, test_res
            torch.cuda.empty_cache()

        del khop_ext, khop_pre

In [ ]:
gc.collect()
torch.cuda.empty_cache()
print('Static k-hop done — GPU memory released.')

## 5. Done

In [ ]:
print('All experiments complete.')
print('Results saved to:')
for d in ['results/benchmark', 'results/benchmark-ppr', 'results/benchmark-khop']:
    n = sum(1 for root, _, files in os.walk(d) if 'full_results.json' in files)
    print(f'  {d}: {n} full_results.json files')
print('\nRun learnable_ppr.ipynb for Learnable PPR results.')
print('Then open benchmark_analysis.ipynb to compare all methods.')